In [2]:
import pandas as pd

income_stmt = pd.read_csv("nike_income_statement_raw.csv", index_col=0)

# ambil baris yang kita perlu untuk margin
profitability = income_stmt.loc[["Total Revenue", "Gross Profit", "Operating Income", "Net Income"]]

# buang kolom (tahun) yang datanya tidak lengkap
profitability = profitability.dropna(axis=1)

# hitung margin dalam persen
margins = pd.DataFrame(index=profitability.columns)
margins["Gross Margin %"] = (profitability.loc["Gross Profit"] / profitability.loc["Total Revenue"]) * 100
margins["Operating Margin %"] = (profitability.loc["Operating Income"] / profitability.loc["Total Revenue"]) * 100
margins["Net Margin %"] = (profitability.loc["Net Income"] / profitability.loc["Total Revenue"]) * 100

margins = margins.round(2)
print(margins)

            Gross Margin %  Operating Margin %  Net Margin %
2026-05-31           42.91                8.18          6.70
2025-05-31           42.73                7.99          6.95
2024-05-31           44.56               12.29         11.10
2023-05-31           43.52               11.55          9.90


In [3]:
print(income_stmt.index.tolist())

['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Net Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Net Non Operating Interest Income Expense', 'Total Other Finance Cost', 'Operating Income', 'Operating Expense', 'Other Operating Expenses', 'Selling General And Administration', 'Selling And Marketing Expense', 'General And Administrative Expense', 'Other Gand A', 'Gross Profit', 'Cost Of Revenue', 'Total Revenue', 'Op

In [4]:
sga = income_stmt.loc["Selling General And Administration"]
revenue = income_stmt.loc["Total Revenue"]

sga_pct = (sga / revenue * 100).round(2)
print(sga_pct)

2026-05-31    34.73
2025-05-31    34.74
2024-05-31    32.27
2023-05-31    31.98
2022-05-31      NaN
dtype: float64


In [5]:
sga_dollars = income_stmt.loc["Selling General And Administration"]
revenue_dollars = income_stmt.loc["Total Revenue"]

comparison = pd.DataFrame({
    "Revenue ($)": revenue_dollars,
    "SG&A ($)": sga_dollars,
})
print(comparison)

             Revenue ($)      SG&A ($)
2026-05-31  4.639800e+10  1.611400e+10
2025-05-31  4.630900e+10  1.608800e+10
2024-05-31  5.136200e+10  1.657600e+10
2023-05-31  5.121700e+10  1.637700e+10
2022-05-31           NaN           NaN


## Analysis: Why Did Operating Margin Fall More Sharply Than Gross Margin?

**Observation:** Gross Margin fell 2 points (45%→43%), but Operating Margin 
fell 4 points (12%→8%) from FY2024 to FY2025 — twice the decline.

**Hypothesis:** If the drop came purely from COGS, the gap between the two 
margins should stay constant. The gap widened from 33 to 35 points → 
suggesting extra pressure from operating expenses (SG&A).

**Verification:** SG&A as % of revenue rose (32%→35%), but in dollar terms 
it actually fell 2.94%. The issue isn't SG&A growing — revenue fell 9.84%, 
far faster than SG&A declined.

**Conclusion:** Operating deleverage — operating costs couldn't be cut as 
fast as revenue was lost, not Nike increasing SG&A spending.

In [6]:
balance_sheet = pd.read_csv("nike_balance_sheet_raw.csv", index_col=0)

net_income = income_stmt.loc["Net Income"]
stockholders_equity = balance_sheet.loc["Stockholders Equity"]
total_assets = balance_sheet.loc["Total Assets"]
total_debt = balance_sheet.loc["Total Debt"]

returns = pd.DataFrame(index=net_income.index)
returns["ROE %"] = (net_income / stockholders_equity * 100)
returns["ROA %"] = (net_income / total_assets * 100)
returns["Debt-to-Equity"] = (total_debt / stockholders_equity)

returns = returns.dropna().round(2)
print(returns)

            ROE %  ROA %  Debt-to-Equity
2026-05-31  20.91   8.09            0.74
2025-05-31  24.36   8.80            0.83
2024-05-31  39.50  14.96            0.83
2023-05-31  36.20  13.51            0.87


In [7]:
revenue = income_stmt.loc["Total Revenue"]
total_assets_matched = balance_sheet.loc["Total Assets"]
stockholders_equity_matched = balance_sheet.loc["Stockholders Equity"]

dupont = pd.DataFrame(index=net_income.index)
dupont["Net Margin"] = net_income / revenue
dupont["Asset Turnover"] = revenue / total_assets_matched
dupont["Equity Multiplier"] = total_assets_matched / stockholders_equity_matched
dupont["ROE (calc) %"] = (dupont["Net Margin"] * dupont["Asset Turnover"] * dupont["Equity Multiplier"] * 100)

dupont = dupont.dropna().round(3)
print(dupont)

            Net Margin  Asset Turnover  Equity Multiplier  ROE (calc) %
2026-05-31       0.067           1.208              2.584        20.908
2025-05-31       0.070           1.266              2.768        24.362
2024-05-31       0.111           1.348              2.641        39.501
2023-05-31       0.099           1.365              2.680        36.204


In [8]:
with pd.option_context('display.float_format', '{:.3f}'.format):
    print(dupont)

            Net Margin  Asset Turnover  Equity Multiplier  ROE (calc) %
2026-05-31       0.067           1.208              2.584        20.908
2025-05-31       0.070           1.266              2.768        24.362
2024-05-31       0.111           1.348              2.641        39.501
2023-05-31       0.099           1.365              2.680        36.204


In [9]:
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
total_liabilities = balance_sheet.loc["Total Liabilities Net Minority Interest"]

leverage_check = pd.DataFrame(index=net_income.index)
leverage_check["Total Debt / Equity"] = total_debt / stockholders_equity_matched
leverage_check["Total Liabilities / Equity"] = total_liabilities / stockholders_equity_matched
leverage_check["Equity Multiplier (Total Liab/Eq + 1)"] = leverage_check["Total Liabilities / Equity"] + 1
leverage_check["Equity Multiplier (actual, Total Assets/Eq)"] = total_assets_matched / stockholders_equity_matched

leverage_check = leverage_check.round(3)
print(leverage_check)

            Total Debt / Equity  Total Liabilities / Equity  \
2026-05-31                0.742                       1.584   
2025-05-31                0.833                       1.768   
2024-05-31                0.828                       1.641   
2023-05-31                0.867                       1.680   
2022-05-31                  NaN                         NaN   

            Equity Multiplier (Total Liab/Eq + 1)  \
2026-05-31                                  2.584   
2025-05-31                                  2.768   
2024-05-31                                  2.641   
2023-05-31                                  2.680   
2022-05-31                                    NaN   

            Equity Multiplier (actual, Total Assets/Eq)  
2026-05-31                                        2.584  
2025-05-31                                        2.768  
2024-05-31                                        2.641  
2023-05-31                                        2.680  
2022-05-31  

## Analysis: DuPont Decomposition — What's Driving the ROE Decline?

**Observation:** ROE fell from 39.5% (FY2024) to 24.4% (FY2025) — a 38.3% 
relative decline.

**Decomposition (ROE = Net Margin × Asset Turnover × Equity Multiplier):**
- Net Margin fell 36.9% (0.111 → 0.070) — the main driver
- Asset Turnover fell 6.1% (1.348 → 1.266) — a minor contributor
- Equity Multiplier actually **rose** 4.8% (2.641 → 2.768)

**Conclusion:** The ROE decline is driven almost entirely by falling 
profitability (Net Margin), not by a change in how Nike is financed. 
Rising leverage (Equity Multiplier) actually cushioned ROE from falling 
even further than 38.3%.

---

## Correction: Nike's Debt-to-Equity Is Not Flat

**Initial mistake:** D/E was briefly concluded to be "flat at 1" over the 
past 4 years — this was wrong, caused by display rounding to 0 decimals 
that happened to round every value to 1.

**Actual figures (3 decimals):** 0.867 (FY2023) → 0.828 (FY2024) → 
0.833 (FY2025) → 0.742 (FY2026) — leverage is slightly declining, not static.

**Why Equity Multiplier (2.6-2.8) is much higher than D/E (~0.8):** D/E only 
counts interest-bearing debt (loans, bonds). Equity Multiplier counts ALL 
liabilities, including non-debt items — accounts payable, accrued expenses, 
etc. Verified mathematically:

`Total Liabilities/Equity + 1 = Total Assets/Equity` — matches exactly 
(2.584, 2.768, 2.641, 2.680 on both sides).

**Lesson:** Display formatting (rounding) can hide real trends in the data. 
Ratios that "look flat" need to be checked against raw figures before being 
treated as fact.

In [10]:
cash_flow = pd.read_csv("nike_cash_flow_raw.csv", index_col=0)
print(cash_flow.index.tolist())

['Free Cash Flow', 'Repurchase Of Capital Stock', 'Repayment Of Debt', 'Issuance Of Debt', 'Capital Expenditure', 'Interest Paid Supplemental Data', 'Income Tax Paid Supplemental Data', 'End Cash Position', 'Beginning Cash Position', 'Effect Of Exchange Rate Changes', 'Changes In Cash', 'Financing Cash Flow', 'Cash Flow From Continuing Financing Activities', 'Net Other Financing Charges', 'Proceeds From Stock Option Exercised', 'Cash Dividends Paid', 'Common Stock Dividend Paid', 'Net Common Stock Issuance', 'Common Stock Payments', 'Net Issuance Payments Of Debt', 'Net Short Term Debt Issuance', 'Net Long Term Debt Issuance', 'Long Term Debt Payments', 'Long Term Debt Issuance', 'Investing Cash Flow', 'Cash Flow From Continuing Investing Activities', 'Net Other Investing Changes', 'Net Investment Purchase And Sale', 'Sale Of Investment', 'Purchase Of Investment', 'Net PPE Purchase And Sale', 'Purchase Of PPE', 'Operating Cash Flow', 'Cash Flow From Continuing Operating Activities', 'C

In [11]:
operating_cf = cash_flow.loc["Operating Cash Flow"]
capex = cash_flow.loc["Capital Expenditure"]
fcf_reported = cash_flow.loc["Free Cash Flow"]

fcf_calculated = operating_cf + capex

check = pd.DataFrame({
    "Operating CF": operating_cf,
    "CapEx": capex,
    "FCF (hitung manual)": fcf_calculated,
    "FCF (dari Yahoo)": fcf_reported,
})
print(check)

                Operating CF            CapEx  FCF (hitung manual)  \
2026-05-31 2,868,000,000.000 -684,000,000.000    2,184,000,000.000   
2025-05-31 3,698,000,000.000 -430,000,000.000    3,268,000,000.000   
2024-05-31 7,429,000,000.000 -812,000,000.000    6,617,000,000.000   
2023-05-31 5,841,000,000.000 -969,000,000.000    4,872,000,000.000   
2022-05-31               NaN              NaN                  NaN   

            FCF (dari Yahoo)  
2026-05-31 2,184,000,000.000  
2025-05-31 3,268,000,000.000  
2024-05-31 6,617,000,000.000  
2023-05-31 4,872,000,000.000  
2022-05-31               NaN  


In [12]:
wc_change = cash_flow.loc["Change In Working Capital"]
inventory_change = cash_flow.loc["Change In Inventory"]

wc_check = pd.DataFrame({
    "Change In Working Capital": wc_change,
    "Change In Inventory": inventory_change,
})
print(wc_check)

            Change In Working Capital  Change In Inventory
2026-05-31         -1,678,000,000.000      -31,000,000.000
2025-05-31           -787,000,000.000      120,000,000.000
2024-05-31            716,000,000.000      908,000,000.000
2023-05-31           -513,000,000.000     -133,000,000.000
2022-05-31                        NaN                  NaN


In [13]:
receivables_change = cash_flow.loc["Change In Receivables"]
payables_change = cash_flow.loc["Change In Payables And Accrued Expense"]

wc_components = pd.DataFrame({
    "Change In Working Capital": wc_change,
    "Change In Inventory": inventory_change,
    "Change In Receivables": receivables_change,
    "Change In Payables": payables_change,
})
print(wc_components)

            Change In Working Capital  Change In Inventory  \
2026-05-31         -1,678,000,000.000      -31,000,000.000   
2025-05-31           -787,000,000.000      120,000,000.000   
2024-05-31            716,000,000.000      908,000,000.000   
2023-05-31           -513,000,000.000     -133,000,000.000   
2022-05-31                        NaN                  NaN   

            Change In Receivables  Change In Payables  
2026-05-31     -1,207,000,000.000    -959,000,000.000  
2025-05-31       -257,000,000.000    -426,000,000.000  
2024-05-31       -329,000,000.000     397,000,000.000  
2023-05-31        489,000,000.000    -225,000,000.000  
2022-05-31                    NaN                 NaN  


## Analysis: What Drove the Working Capital Swing in FY2025?

**Initial hypothesis:** Inventory buildup was assumed to be the main driver 
behind the sharp Operating Cash Flow decline in FY2025.

**Correction:** Looking at FY2025 in isolation, Change in Inventory was 
actually +120M (positive — inventory decreased, releasing cash). The 
initial hypothesis was wrong; the right comparison is the year-over-year 
**swing**, not a single year's value in isolation.

**Verification (FY2024 → FY2025 swing):**
- Inventory: +908M → +120M = swing of **-788M**
- Payables: +397M → -426M = swing of **-823M**
- Receivables: -329M → -257M = swing of **+72M** (partially offsetting, 
  not worsening)

Combined swing (-1,539M) closely matches the actual reported Change in 
Working Capital swing (-1,503M), confirming these are the right components.

**Conclusion:** Inventory and Payables are **co-equal drivers**, roughly 
matched in magnitude — not one dominant factor as first assumed. Payables 
reversed from a cash source (delaying supplier payments) to a cash use 
(paying suppliers faster), while Inventory released far less cash than 
the prior year (goods moved more slowly, consistent with softer sales). 
Receivables actually cushioned the decline rather than adding to it.

**Why this matters for the FCF projection:** because working capital 
changes are generally driven by timing decisions (payment terms, inventory 
levels) rather than structural cost changes, they are more likely to 
**mean-revert** than SG&A stickiness is. This supports treating recent FCF 
weakness as partly temporary — one of the reasons a range of scenarios 
(Bear/Base/Bull), rather than a single extrapolated number, was used for 
the FCF margin assumption.

In [14]:
revenue = income_stmt.loc["Total Revenue"]
fcf_margin = (fcf_reported / revenue * 100).round(2)
print(fcf_margin)

2026-05-31    4.710
2025-05-31    7.060
2024-05-31   12.880
2023-05-31    9.510
2022-05-31      NaN
dtype: float64


In [15]:
base_revenue = revenue["2026-05-31"]
growth_rate = 0.03

years = [2027, 2028, 2029, 2030, 2031]
projected_revenue = []
rev = base_revenue
for year in years:
    rev = rev * (1 + growth_rate)
    projected_revenue.append(rev)

fcf_margins = {
    "Bear": 0.05885,
    "Base": 0.072,
    "Bull": 0.0854,
}

projection = pd.DataFrame(index=years)
projection["Revenue"] = projected_revenue
for scenario, margin in fcf_margins.items():
    projection[f"FCF ({scenario})"] = projection["Revenue"] * margin

print(projection.round(0))

                Revenue        FCF (Bear)        FCF (Base)        FCF (Bull)
2027 47,789,940,000.000 2,812,437,969.000 3,440,875,680.000 4,081,260,876.000
2028 49,223,638,200.000 2,896,811,108.000 3,544,101,950.000 4,203,698,702.000
2029 50,700,347,346.000 2,983,715,441.000 3,650,425,009.000 4,329,809,663.000
2030 52,221,357,766.000 3,073,226,905.000 3,759,937,759.000 4,459,703,953.000
2031 53,787,998,499.000 3,165,423,712.000 3,872,735,892.000 4,593,495,072.000


In [16]:
current_price = 36.03  # manual, per 16 Sept 2026 (yfinance company.info gagal)
beta = 1.10             # manual, sumber sama

shares_outstanding = income_stmt.loc["Diluted Average Shares"]["2026-05-31"]
market_cap = current_price * shares_outstanding

print(f"Shares Outstanding (FY2026): {shares_outstanding:,.0f}")
print(f"Market Cap (dihitung): {market_cap:,.0f}")

pd.DataFrame([{
    "current_price": current_price,
    "beta": beta,
    "shares_outstanding": shares_outstanding,
    "market_cap": market_cap,
}]).to_csv("nike_market_data_manual.csv", index=False)
print("✓ Disimpan ke CSV")

Shares Outstanding (FY2026): 1,481,000,000
Market Cap (dihitung): 53,360,430,000
✓ Disimpan ke CSV


In [17]:
interest_paid = cash_flow.loc["Interest Paid Supplemental Data"]
total_debt = balance_sheet.loc["Total Debt"]
tax_rate = income_stmt.loc["Tax Rate For Calcs"]

cost_of_debt_check = pd.DataFrame({
    "Interest Paid": interest_paid,
    "Total Debt": total_debt,
    "Tax Rate": tax_rate,
})
print(cost_of_debt_check)

             Interest Paid         Total Debt  Tax Rate
2026-05-31 323,000,000.000 11,033,000,000.000     0.203
2025-05-31 389,000,000.000 11,013,000,000.000     0.171
2024-05-31 381,000,000.000 11,952,000,000.000     0.149
2023-05-31 347,000,000.000 12,144,000,000.000     0.182
2022-05-31             NaN                NaN       NaN


In [18]:
wacc = 0.0844
terminal_growth = 0.03

results = {}

for scenario in ["Bear", "Base", "Bull"]:
    fcf_series = projection[f"FCF ({scenario})"]
    
    # diskon tiap tahun FCF ke nilai sekarang
    pv_fcf = sum(fcf_series.iloc[i] / (1 + wacc) ** (i + 1) for i in range(len(fcf_series)))
    
    # terminal value, dihitung dari FCF tahun terakhir (2031)
    fcf_final_year = fcf_series.iloc[-1]
    terminal_value = fcf_final_year * (1 + terminal_growth) / (wacc - terminal_growth)
    pv_terminal_value = terminal_value / (1 + wacc) ** len(fcf_series)
    
    enterprise_value = pv_fcf + pv_terminal_value
    
    # dari enterprise value ke equity value: kurangi net debt
    cash = balance_sheet.loc["Cash And Cash Equivalents"]["2026-05-31"]
    net_debt = total_debt["2026-05-31"] - cash
    equity_value = enterprise_value - net_debt
    
    implied_price = equity_value / shares_outstanding
    
    results[scenario] = {
        "PV of FCF": pv_fcf,
        "Terminal Value": terminal_value,
        "PV of Terminal Value": pv_terminal_value,
        "Enterprise Value": enterprise_value,
        "Net Debt": net_debt,
        "Equity Value": equity_value,
        "Implied Share Price": implied_price,
    }

results_df = pd.DataFrame(results).round(2)
print(results_df)

                                   Bear               Base               Bull
PV of FCF            11,730,287,273.200 14,351,413,486.320 17,022,370,996.280
Terminal Value       59,933,573,952.920 73,325,697,954.290 86,972,425,073.560
PV of Terminal Value 39,968,940,098.130 48,899,977,690.150 58,000,806,871.370
Enterprise Value     51,699,227,371.320 63,251,391,176.470 75,023,177,867.650
Net Debt              3,470,000,000.000  3,470,000,000.000  3,470,000,000.000
Equity Value         48,229,227,371.320 59,781,391,176.470 71,553,177,867.650
Implied Share Price              32.570             40.370             48.310


## DCF Valuation Summary

**Revenue assumption:** Projected from the FY2026 base (~$46.4B) at a flat 
**3% annual growth rate**. Revenue pattern was flat → single sharp shock 
(-9.84%, FY2024→FY2025) → flat again (FY2025→FY2026), which supports 
treating the current level as a new stable base rather than assuming 
continued decline or a rebound to FY2023-24 levels.

**FCF Margin assumption — three scenarios, not a single point estimate:**
- **Bear (5.885%):** average of the last 2 years only — assumes no 
  recovery, SG&A stickiness persists
- **Bull (8.54%):** average of all 4 years — assumes working capital *and* 
  SG&A both normalize back toward historical levels
- **Base (7.2%):** midpoint between the two

A single-point estimate was deliberately avoided because the FCF decline 
was driven by two factors with different characters: SG&A stickiness 
(structural, needs active management decisions to improve) and working 
capital swings (more likely to mean-revert on their own).

**Discount rate — WACC = 8.44%:**
- Cost of Equity = 9.7% (CAPM: risk-free rate 4.2%, beta 1.10, equity risk 
  premium 5%)
- Cost of Debt (after-tax) = 2.33% (pre-tax 2.93%, effective tax rate 
  ~20.3%)
- Weights: 82.87% equity / 17.13% debt (market cap $53.36B vs. total debt 
  $11.03B)
- Terminal growth rate: 3% (Gordon Growth Model)

**Results — Implied share price:**

| Scenario | Implied Share Price |
|----------|---------------------|
| Bear     | $32.57 |
| Base     | $40.37 |
| Bull     | $48.31 |

**Comparison to market price:** Nike traded at **$36.03** as of September 
16, 2026 (down ~79% from its 52-week high). This sits about **44%** of the 
way from Bear to Base — slightly closer to the Bear case.

**Interpretation:** the market's pricing is broadly more consistent with 
the operating deleverage thesis (SG&A stickiness, which doesn't 
self-correct) than with the working capital swings (which are more likely 
to normalize on their own) found in the historical analysis above.

**Limitation:** this model is built entirely from historical financial 
statement data. Market price reflects a much wider information set — 
competitive dynamics, execution in China, forward management guidance, 
analyst sentiment — none of which this model captures. Consistent with 
that uncertainty, published analyst price targets for NKE span an 
unusually wide range (roughly $23 to $75 as of September 2026), reflecting 
genuine disagreement in the market itself, not just uncertainty in this 
specific model. The conclusion here should be read as "consistent with," 
not "proof of," the operating deleverage thesis.

In [19]:
import time
import yfinance as yf

def fetch_with_retry(ticker_symbol, max_retries=3, wait_seconds=30):
    for attempt in range(max_retries):
        try:
            ticker = yf.Ticker(ticker_symbol)
            info = ticker.info
            if info and info.get("shortName"):
                return info
            else:
                print(f"  {ticker_symbol}: data kosong, percobaan {attempt+1}/{max_retries}")
        except Exception as e:
            print(f"  {ticker_symbol}: error ({type(e).__name__}), percobaan {attempt+1}/{max_retries}")
        
        if attempt < max_retries - 1:
            print(f"  Menunggu {wait_seconds} detik sebelum coba lagi...")
            time.sleep(wait_seconds)
    
    return None

peers = ["ADS.DE", "UAA", "LULU", "PUM.DE"]
peer_metrics = {}

for symbol in peers:
    print(f"Fetching {symbol}...")
    info = fetch_with_retry(symbol)
    if info:
        peer_metrics[symbol] = {
            "name": info.get("shortName"),
            "price": info.get("currentPrice"),
            "trailingEPS": info.get("trailingEps"),
            "marketCap": info.get("marketCap"),
            "totalDebt": info.get("totalDebt"),
            "totalCash": info.get("totalCash"),
            "ebitda": info.get("ebitda"),
        }
        print(f"  ✓ {info.get('shortName')}")
    else:
        print(f"  ✗ Gagal setelah beberapa percobaan")
    time.sleep(10)

print(peer_metrics)

Fetching ADS.DE...
  ✓ adidas AG                     N
Fetching UAA...
  ✓ Under Armour, Inc.
Fetching LULU...
  ✓ lululemon athletica inc.
Fetching PUM.DE...
  ✓ PUMA SE                       I
{'ADS.DE': {'name': 'adidas AG                     N', 'price': 143.8, 'trailingEPS': 7.8, 'marketCap': 25174042624, 'totalDebt': 5977999872, 'totalCash': 1562000000, 'ebitda': 2664000000}, 'UAA': {'name': 'Under Armour, Inc.', 'price': 4.53, 'trailingEPS': -1.1, 'marketCap': 1945841408, 'totalDebt': 1376077056, 'totalCash': 395980992, 'ebitda': 173035008}, 'LULU': {'name': 'lululemon athletica inc.', 'price': 102.28, 'trailingEPS': 12.15, 'marketCap': 11323420672, 'totalDebt': 2141127040, 'totalCash': 1389736960, 'ebitda': 2383926016}, 'PUM.DE': {'name': 'PUMA SE                       I', 'price': 22.36, 'trailingEPS': -3.01, 'marketCap': 3291591424, 'totalDebt': 2657299968, 'totalCash': 372500000, 'ebitda': -17000000}}


In [20]:
import pandas as pd

comp_table = pd.DataFrame(peer_metrics).T

comp_table["EV"] = comp_table["marketCap"] + comp_table["totalDebt"] - comp_table["totalCash"]

comp_table["P/E"] = comp_table.apply(
    lambda row: round(row["price"] / row["trailingEPS"], 2) if row["trailingEPS"] > 0 else "N/M",
    axis=1
)
comp_table["EV/EBITDA"] = comp_table.apply(
    lambda row: round(row["EV"] / row["ebitda"], 2) if row["ebitda"] > 0 else "N/M",
    axis=1
)

print(comp_table[["name", "P/E", "EV/EBITDA"]])

                                   name    P/E EV/EBITDA
ADS.DE  adidas AG                     N 18.440    11.110
UAA                  Under Armour, Inc.    N/M    16.910
LULU           lululemon athletica inc.  8.420     5.070
PUM.DE  PUMA SE                       I    N/M       N/M


In [21]:
nike_eps = income_stmt.loc["Diluted EPS"]["2026-05-31"]
nike_ebitda = income_stmt.loc["EBITDA"]["2026-05-31"]
nike_ev = market_cap + total_debt["2026-05-31"] - cash

nike_pe = round(current_price / nike_eps, 2)
nike_ev_ebitda = round(nike_ev / nike_ebitda, 2)

print(f"Nike P/E: {nike_pe}")
print(f"Nike EV/EBITDA: {nike_ev_ebitda}")

# rata-rata peer, KECUALI yang N/M
valid_pe = [v for v in comp_table["P/E"] if v != "N/M"]
valid_ev_ebitda = [v for v in comp_table["EV/EBITDA"] if v != "N/M"]

peer_avg_pe = round(sum(valid_pe) / len(valid_pe), 2)
peer_avg_ev_ebitda = round(sum(valid_ev_ebitda) / len(valid_ev_ebitda), 2)

print(f"\nRata-rata peer P/E (n={len(valid_pe)}): {peer_avg_pe}")
print(f"Rata-rata peer EV/EBITDA (n={len(valid_ev_ebitda)}): {peer_avg_ev_ebitda}")

Nike P/E: 17.16
Nike EV/EBITDA: 12.37

Rata-rata peer P/E (n=2): 13.43
Rata-rata peer EV/EBITDA (n=3): 11.03


Synthesis: Reconciling DCF and Comps

DCF compares Nike to its own historical FCF trajectory, while Comps compares Nike to its competitors at a single point in time — these are not contradictory. Nike's own profitability declined because SG&A costs didn't shrink proportionally with falling profit (operating deleverage, detailed above) — this is what drives the DCF's pessimistic valuation relative to Nike's own historical performance. However, relative to its competitors, Nike's decline is far milder: Under Armour posted a net loss (negative EPS), and Puma posted a negative EBITDA — an operating loss, not just a net loss. This is consistent with a broader apparel industry downturn: Nike isn't an exception to the weakness, but it's weathering it noticeably better than its peers.